# **DIPY**
**Conclusion:**
The best signal to noise ratio (SNR) was found by first denoising using mpPCA -> into gibbs correction.
- Further adjustment such as motion correction only worsened the SNR (see SNR_investi.ipynb script). 
- dwMRI was then registered to the t1w structural scans. 

In [ ]:
#Homemade Functions
from functions.functions_analysis import *
from functions.Path_combine_function import *
from functions.plot_functions import *
from functions.Preproc_functions import *

#More nifti packages
from nilearn import plotting
import nibabel as nib

# Dipy
import dipy as dp
from dipy.io.image import load_nifti, save_nifti
from dipy.io import read_bvals_bvecs
from dipy.core.gradients import gradient_table
import dipy.data as dpd

#Preproc
from dipy.align import motion_correction
import dipy.direction.peaks as dpp
from dipy.viz import window, actor
from dipy.segment.mask import median_otsu
from dipy.core.histeq import histeq

from dipy.denoise.nlmeans import nlmeans
from dipy.denoise.noise_estimate import estimate_sigma

#DIPY Plot
from dipy.viz import window, actor
from dipy.data import get_sphere

#Regular Packages
import keyboard  # For detecting keypresses
import IPython

import numpy as np
import os
import ants

from pathlib import Path
from time import time
import time  # For simulating work
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import scipy.io

# Preprocessing dwMRI

In [ ]:
########## Start with preprocessing: #################
### Denoise 
DTI_paths = PathFinder("Dti_SE.nii")
noise_dict = mppca_denoise(file_paths = DTI_paths, output_name = "denoised_affine_ok")

### Gibbs Correction
DTI_paths = PathFinder("denoised_affine_ok")
gibbs_correct(file_paths= DTI_paths, output_name= "gibbs_denoised_affine_ok")

In [ ]:
######## Combine with cosine blend ############

# Pre-Proc
DTI_paths = PathFinder("gibbs_denoised_affine_ok.nii")
Combined_DTI_proproc_blend = Combine_NIFTI_blend(file_paths= DTI_paths, overlap_slices= 10 , blend= "cosine")
ants.image_write(Combined_DTI_proproc_blend, "Combined_Data/DTI_combined_preproc_blend.nii.gz")

# Non Pre-proc 
file_paths = PathFinder("Dti_SE.nii")
Combined_DTI_blend = Combine_NIFTI_blend(file_paths = file_paths, overlap_slices= 10 , blend= "cosine")
ants.image_write(Combined_DTI_blend, "Combined_Data/DTI_combined_blend.nii.gz")

####### Transpose the images and Save #######
# Pre-Proc
Combined_DTI_proproc_blend_transposed = np.transpose(Combined_DTI_proproc_blend.numpy(), axes = (1,0,2,3))
Combined_DTI_proproc_blend_transposed_ants = ants.from_numpy(Combined_DTI_proproc_blend_transposed, origin = Combined_DTI_proproc_blend.origin, spacing = Combined_DTI_proproc_blend.spacing, direction = Combined_DTI_proproc_blend.direction)
ants.image_write(Combined_DTI_proproc_blend_transposed_ants , "Combined_Data/Transposed/DTI_combined_preproc_blend_trans.nii.gz")

# Non Pre-Proc
Combined_DTI_blend_transposed = np.transpose(Combined_DTI_blend.numpy(), axes = (1,0,2,3))
Combined_DTI_blend_transposed_ants = ants.from_numpy(Combined_DTI_blend_transposed, origin = Combined_DTI_blend.origin, spacing = Combined_DTI_blend.spacing, direction = Combined_DTI_blend.direction)
ants.image_write(Combined_DTI_blend_transposed_ants , "Combined_Data/Transposed/DTI_combined_blend_trans.nii.gz")

In [ ]:
####################### Bias Field Correction on All Files ####################### 
DTI_denoised_gibbs_paths = PathFinder("gibbs_denoised_affine_ok")


for path in DTI_denoised_gibbs_paths:
    file_path_parent = Path(path).parent
    outpath = os.path.join(file_path_parent, "DTI_SE_BFC_gibbs_denoised_affine_ok.nii.gz")

    print(f"Starting processing for {path}")
    # Apply N4 bias field correction
    DTI_file = ants.image_read(path)

    b0 = DTI_file[:,:,:,0:3]
    b0_mean = np.mean(b0, axis=3)

    direction_clean = np.array(DTI_file.direction[:3, :3], dtype=np.float64, order='C')
    #print("Shape:", direction_clean.shape)
    #print("Contiguous:", direction_clean.flags['C_CONTIGUOUS'])

    b0_mean_ants = ants.from_numpy(
        b0_mean,
        origin= DTI_file.origin[:3],
        spacing= DTI_file.spacing[:3],
        direction= direction_clean
    )

    print("Starting N4 bias field correction...")

    bias_field = ants.n4_bias_field_correction(b0_mean_ants,
                                                    shrink_factor=1,  # Adjust shrink factor for speed vs accuracy
                                                    mask=None,  # Use the mask if available, or set to None
                                                    convergence= {'iters': [50, 50, 50,30], 'tol': 1e-6},
                                                    spline_param= [1,1,1],  # Adjust spline parameter for bias field smoothness
                                                    return_bias_field=True,  # Set to True if you want to return the bias field
                                                    verbose=False)  # Set to True for detailed output
    # Create a new array to store the corrected DTI
    DTI_corrected = np.empty_like(DTI_file.numpy())

    # Apply the bias field to each volume (last dimension)
    for i in range(DTI_file.shape[3]):
        DTI_corrected[:, :, :, i] = DTI_file.numpy()[:, :, :, i] / bias_field.numpy()

    print(f"Corrected DTI shape: {DTI_corrected.shape}, Original DTI shape: {DTI_file.shape}")

    # Save the corrected image
    DTI_corrected_ants = ants.from_numpy(DTI_corrected, origin=DTI_file.origin,
                                          spacing=DTI_file.spacing, direction=DTI_file.direction)
    ants.image_write(DTI_corrected_ants, outpath)
    np.save(os.path.join(file_path_parent, "bias_field_2_2_2.npy"), bias_field.numpy())


# Preprocessing Structural

In [ ]:
#Structural Files Solo
t2_paths = PathFinder("RARE_2D_Ax.nii")


for path in t2_paths:
    file_path_parent = Path(path).parent
    outpath = os.path.join(file_path_parent, "RARE_2D_Ax_preproc.nii.gz")

    print(f"Starting processing for {path}")

    #check if the file exists
    if os.path.exists(outpath):
        print(f"File already exists: {outpath}")
        continue  # Skip processing if the file already exists

    #Load the nifti files
    RARE_file, RARE_affine = load_nifti(path)
    RARE_file = RARE_file.astype(np.float32)  # Convert to float32 to reduce memory usage

    #Mask
    print("Starting masking...")
    RARE_combined_masked, mask = median_otsu(RARE_file, median_radius=2, numpass=2,
                                        autocrop=False, dilate=None)
    np.save(os.path.join(file_path_parent,"RARE_mask.npy"), mask)
    
    ##### Denoise ###
    print("Starting denoising...")
    # Estimate noise standard deviation
    sigma = estimate_sigma(RARE_file)

    # Apply Non-Local Means (NLM) denoising
    denoised_data = nlmeans(RARE_file, sigma = sigma, patch_radius=2, block_radius=5, mask = mask)

    ### Gibbs Correction ####
    print("Starting Gibbs correction...")
    data_corrected = gibbs_removal(denoised_data, slice_axis=2, num_processes=-1)


    #Save the data
    print("Saving data...")
    RARE_data_nifti = nib.Nifti1Image(data_corrected,   affine = RARE_affine)

    save_nifti(outpath, RARE_data_nifti.get_fdata(),
        RARE_data_nifti.affine)



In [ ]:
######## Combine with cosine blend ############

# Non Pre-Proc
file_paths = PathFinder("RARE_2D_Ax.nii")
Combined_rare_blend = Combine_NIFTI_blend(file_paths= file_paths, overlap_slices= 10 , blend= "cosine")
ants.image_write(Combined_rare_blend, "Combined_Data/RARE_combined_blend.nii.gz")

# Pre-proc 
file_paths = PathFinder("RARE_2D_Ax_preproc.nii")
Combined_rare_preproc_blend = Combine_NIFTI_blend(file_paths= file_paths, overlap_slices= 10 , blend= "cosine")
ants.image_write(Combined_rare_preproc_blend, "Combined_Data/RARE_preproc_combined_blend.nii.gz")



#### Transpose ###
# Non Pre-proc
Combined_rare_blend_transposed = np.transpose(Combined_rare_blend.numpy(), axes = (1,0,2))
Combined_rare_blend_transposed_ants = ants.from_numpy(Combined_rare_blend_transposed, 
                                                      origin = Combined_rare_blend.origin, 
                                                      spacing = Combined_rare_blend.spacing, direction = 
                                                      Combined_rare_blend.direction)

ants.image_write(Combined_rare_blend_transposed_ants , "Combined_Data/Transposed/RARE_combined_blend_trans.nii.gz")


#preproc
Combined_rare_preproc_blend_transposed = np.transpose(Combined_rare_preproc_blend.numpy(), axes = (1,0,2))
Combined_rare_preproc_blend_transposed_ants = ants.from_numpy(Combined_rare_preproc_blend_transposed,
                                                               origin = Combined_rare_preproc_blend.origin, 
                                                               spacing = Combined_rare_preproc_blend.spacing, 
                                                               direction = Combined_rare_preproc_blend.direction)

ants.image_write(Combined_rare_preproc_blend_transposed_ants , "Combined_Data/Transposed/RARE_preproc_combined_blend_trans.nii.gz")

In [ ]:
####################### Bias Field Correction on All Files ####################### 
t2_paths = PathFinder("RARE_2D_Ax_preproc.nii")
for path in t2_paths:
    file_path_parent = Path(path).parent
    outpath = os.path.join(file_path_parent, "RARE_2D_Ax_preproc_BFC.nii.gz")

    print(f"Starting processing for {path}")

    #check if the file exists
    #if os.path.exists(outpath):
    #    print(f"File already exists: {outpath}")
    #    continue  # Skip processing if the file already exist

    # Apply N4 bias field correction
    RARE_file = ants.image_read(path)
    print("Starting N4 bias field correction...")

    bias_field = ants.n4_bias_field_correction(RARE_file,
                                                    shrink_factor=1,  # Adjust shrink factor for speed vs accuracy
                                                    mask=None,  # Use the mask if available, or set to None
                                                    convergence= {'iters': [50, 50, 50,30], 'tol': 1e-6},
                                                    spline_param= [2,2,2],  # Adjust spline parameter for bias field smoothness
                                                    return_bias_field=True,  # Set to True if you want to return the bias field
                                                    verbose=False)  # Set to True for detailed output
    
    RARE_corrected = RARE_file / bias_field


    # Save the corrected image
    ants.image_write(RARE_corrected, outpath)
    np.save(os.path.join(file_path_parent, "bias_field_2_2_2.npy"), bias_field.numpy())